# 01 · Dataset

Genera los ejemplos sintéticos, valida todo el dataset y arma los splits estratificados por `prioridad`. Los ejemplos `manual`/`real` los escribe el equipo a mano.

> **Ojo:** los splits ya están generados y versionados en `data/splits/` (incluyen los 300 ejemplos manuales). No hace falta correr este notebook antes de los experimentos; si se corre, regenera los sintéticos y los splits.

In [ ]:
# === Setup en Google Colab (ejecutar solo en Colab) ===
# Descomenta estas líneas la primera vez en Colab.
# Rama con los splits que incluyen los datos manuales (cambiar a main cuando se fusione):
# !git clone -b splits-con-manual https://github.com/Hammonne/triaje-clinico-lora.git
# %cd triaje-clinico-lora
# Colab trae torchao 0.10, que peft 0.21 rechaza (pide >= 0.16) y no usamos:
# !pip uninstall -q -y torchao
# Colab ya trae todas las dependencias; estas líneas solo verifican mínimos e instalan el paquete:
# !pip install -q -r requirements.txt
# !pip install -q -e .
# from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# === Celda de configuración (única fuente de parámetros) ===
import sys; sys.path.insert(0, "src")
from pathlib import Path
from triaje.config import Config, set_seed

config = Config()
# En Colab, apuntar las salidas a Drive para sobrevivir desconexiones:
# config.drive_dir = Path("/content/drive/MyDrive/triaje")
# config.results_dir = config.drive_dir / "results"
# config.output_dir  = config.drive_dir / "adaptadores"
set_seed(config.seed)
print("run_id ejemplo:", config.run_id(), "| modelo:", config.model_name)

## Generar ejemplos sintéticos

Solo `origen: sintetico`. El 20% manual lo escribe el equipo.

In [ ]:
!python scripts/generar_sintetico.py --n 1200 --seed 42 --salida data/sintetico/lote01.jsonl

## Validar todo el dataset (manual + real + sintetico)

In [ ]:
!python scripts/validar_dataset.py data/sintetico/*.jsonl data/manual/*.jsonl

## Construir y guardar los splits (estratificados, semilla fija)

In [ ]:
from triaje.data import cargar_todos, dividir, guardar_splits
ejemplos = cargar_todos(config.data_dir)
splits = dividir(ejemplos, config.frac_train, config.frac_val, config.frac_test, seed=config.seed_split)
print(splits.resumen())
guardar_splits(splits, config.data_dir / "splits")